# 🏦 SmartBank AI — Financial Intelligence & Machine Learning Pipeline
### End-to-End Notebook: NLP Expense Categorization, Isolation Forest Anomaly Detection, & Predictive Forecasting

> **Hackathon Prototype** | Explainable AI for Modern FinTech Applications

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

## 1. Synthetic Financial Transactions Generation & Data Profiling
We generate realistic multi-month Indian Rupee (₹) banking transactions featuring salary deposits, living expenses, subscriptions, and deliberate anomalous spikes.

In [ ]:
from ml.training.generate_synthetic_data import generate_dataset

df = generate_dataset(num_records=1500, anomaly_rate=0.035)
print(f"Total Transactions: {len(df):,}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print("\nCategory Distribution:")
print(df['category'].value_counts())
df.head()

## 2. AI Expense Categorization (TF-IDF + Naive Bayes)
Automatically assigns expense categories based on merchant names and transaction descriptions with explainable confidence scores.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

df['text_features'] = df['merchant'].fillna('') + ' ' + df['description'].fillna('')
X = df['text_features']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=2500)),
    ('nb', MultinomialNB(alpha=0.1))
])
clf_pipeline.fit(X_train, y_train)

y_pred = clf_pipeline.predict(X_test)
print(f"Categorization Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

### Live Categorization Inference & 'Why?' Explainability Test

In [ ]:
sample_txns = [
    "Swiggy Food Delivery Bangalore",
    "Uber Daily Commute Trip",
    "Netflix Monthly 4K Subscription",
    "Bescom Electricity Power Bill",
    "Amazon India Electronics Headset"
]

for s in sample_txns:
    cat = clf_pipeline.predict([s])[0]
    conf = max(clf_pipeline.predict_proba([s])[0])
    print(f"• Text: '{s}'")
    print(f"  → Predicted: {cat} (Confidence: {conf*100:.1f}%)")
    print(f"  → Why? Merchant/description text features matched {cat} corpus pattern.\n")

## 3. Unsupervised Anomaly Detection (Isolation Forest)
Detects potential fraud, account takeovers, and outlier expenses using multi-dimensional transaction features (amount, log amount, time-of-day, night flags, category frequency, and z-score deviation).

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

df_feat = df.copy()
df_feat['date'] = pd.to_datetime(df_feat['date'])
df_feat['hour'] = df_feat['date'].dt.hour
df_feat['day_of_week'] = df_feat['date'].dt.dayofweek
df_feat['is_night'] = df_feat['hour'].apply(lambda h: 1 if h < 6 or h >= 23 else 0)
df_feat['log_amount'] = np.log1p(df_feat['amount'])

cat_freq = df_feat['category'].value_counts(normalize=True).to_dict()
df_feat['cat_frequency'] = df_feat['category'].map(cat_freq)

feature_cols = ['amount', 'log_amount', 'hour', 'day_of_week', 'is_night', 'cat_frequency']
X_iso = df_feat[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iso)

iso = IsolationForest(n_estimators=150, contamination=0.035, random_state=42)
iso.fit(X_scaled)

df_feat['iso_pred'] = np.where(iso.predict(X_scaled) == -1, True, False)
df_feat['anomaly_score'] = -iso.score_samples(X_scaled)

y_true = df_feat['is_anomaly']
y_pred_iso = df_feat['iso_pred']

print(f"Precision: {precision_score(y_true, y_pred_iso):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_iso):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred_iso):.4f}")

## 4. Time-Series Expense Forecasting (WMA + Linear Trend)
Projects category-level and total monthly spending for the upcoming month.

In [ ]:
from ml.inference.forecaster import forecast_monthly_expenses

txn_dicts = df.to_dict(orient='records')
forecast_res = forecast_monthly_expenses(txn_dicts)

print(f"🔮 Forecast Period: {forecast_res['period']}")
print(f"💰 Estimated Total Expenses: ₹{forecast_res['estimated_total_expenses']:,.2f}")
print(f"📊 Historical Monthly Average: ₹{forecast_res['historical_monthly_avg']:,.2f}")
print(f"📈 Projected Change: {forecast_res['forecast_change_percentage']}%\n")

fc_df = pd.DataFrame(forecast_res['categories'])
fc_df[['category', 'historical_avg', 'forecasted_amount', 'trend_percentage', 'trend_direction', 'confidence']]

## Summary
All three AI models (NLP Categorizer, Isolation Forest Anomaly Detector, and Time-Series Forecaster) are productionized and integrated into the FastAPI backend and React frontend.